# Loan Data Processing — Assignment Five (Part B1)

This notebook reproduces `code/loan-data-processing.py` from Lesson 5, step by
step, typed into notebook cells with the same section numbering and comments as
the class script. The only required change is in **Step 10**: a different scaler
is used instead of `StandardScaler`.


In [72]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler  # chosen instead of StandardScaler -- see Step 10

print("Successfully imported libraries.")

Successfully imported libraries.


## STEP 1 - Load + initial snapshot

In [73]:
CSV_PATH = "raw_loan_dataset.csv"
OUT_PATH = "clean_loan_dataset.csv"

df = pd.read_csv(CSV_PATH)

print("\n=== INITIAL HEAD ===")
print(df.head())


=== INITIAL HEAD ===
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  


In [74]:
print("\n=== INITIAL INFO ===")
print(df.info())


=== INITIAL INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB
None


In [75]:
print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())


=== INITIAL MISSING VALUES ===
Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


## STEP 2 - Clean currency formatting

In [76]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

print(df[["Income", "LoanAmount"]].head())

     Income  LoanAmount
0  108810.0     25800.0
1   96482.0     11200.0
2   28478.0     12100.0
3   25851.0      7000.0
4   38396.0     10700.0


## STEP 3 - Normalize categorical typos BEFORE imputation

In [77]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

print(df["HasCollateral"].value_counts(dropna=False))
print(df["PreviousDefaults"].value_counts(dropna=False))
print(df["Approved"].value_counts(dropna=False))

HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64
PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64
Approved
Yes    68
No     35
Name: count, dtype: int64


## STEP 4 - Impute missing values

In [78]:
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

print(df.isnull().sum())

Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


## STEP 5 - Remove duplicates

In [79]:
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


## STEP 6 - IQR capping on numeric columns
L3: clip extreme values to the IQR fence -- same idea as house data-processing.py


In [80]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

print(df[["Income", "CreditScore", "LoanAmount", "EmploymentYears"]].describe())

             Income  CreditScore    LoanAmount  EmploymentYears
count     100.00000   100.000000    100.000000       100.000000
mean    72220.22625   651.100000  25507.125000        12.654500
std     29179.39312    96.218239  14793.836616         7.011164
min     25851.00000   484.000000   5000.000000         0.500000
25%     47964.75000   576.000000  14400.000000         6.275000
50%     69460.50000   638.000000  20700.000000        12.550000
75%     95826.50000   730.500000  35125.000000        17.975000
max    167619.12500   920.000000  66212.500000        25.000000


## STEP 7 - Label encoding (Yes/No -> 0/1)
L3: same mapping for target (Approved) and two-category features. Approved=1, Rejected=0


In [81]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))


=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


## STPE 8 - Class balance check
L3/L5: imbalanced classes can make Accuracy misleading


In [82]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes -- consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


## STEP 9 - Feature engineering (no leakage from label)

In [83]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

print(df[["DebtToIncome", "IncomePerYearEmployed"]].describe())

       DebtToIncome  IncomePerYearEmployed
count    100.000000             100.000000
mean       0.360658            8767.039052
std        0.158585           10230.239099
min        0.033409            1112.975207
25%        0.236715            3674.498252
50%        0.352374            5161.894371
75%        0.495593            9312.565407
max        0.822050           61041.111111


## STEP 10 - Feature Scaling

Instead of using `StandardScaler` like the class example, I used **`RobustScaler`** for this assignment.

I chose `RobustScaler` because it uses the **median** and **IQR**, making it better for handling loan data with remaining outliers and skewed values, especially in features like `Income` and `LoanAmount`.

Only the numeric features were scaled. The target column (`Approved`) and binary columns (`HasCollateral` and `PreviousDefaults`) were not scaled.

In [84]:
binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print(df[scale_cols].describe())


           Income  CreditScore  EmploymentYears  LoanAmount  DebtToIncome  \
count  100.000000   100.000000       100.000000  100.000000  1.000000e+02   
mean     0.057660     0.084790         0.008932    0.231948  3.199876e-02   
std      0.609660     0.622772         0.599245    0.713816  6.125864e-01   
min     -0.911156    -0.996764        -1.029915   -0.757539 -1.232107e+00   
25%     -0.449122    -0.401294        -0.536325   -0.303981 -4.467717e-01   
50%      0.000000     0.000000         0.000000    0.000000 -1.075529e-16   
75%      0.550878     0.598706         0.463675    0.696019  5.532283e-01   
max      2.050878     1.825243         1.064103    2.196019  1.814275e+00   

       IncomePerYearEmployed  
count           1.000000e+02  
mean            6.394292e-01  
std             1.814494e+00  
min            -7.181396e-01  
25%            -2.638131e-01  
50%             8.153200e-17  
75%             7.361869e-01  
max             9.911059e+00  


## STEP 11 - Final snapshot + save
Features (X) = all columns except Approved; label (y) = Approved


In [85]:
print("\n=== FINAL HEAD ===")
print(df.head())



=== FINAL HEAD ===
     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445244               8.274536  
1                 0         1     -0.912749               0.153994  
2                 0         1      0.280113              -0.126321  
3                 0         1     -0.315175              -0.669033  
4                 0         1     -0.284688              -0.284975  


In [86]:
print("\n=== FINAL INFO ===")
print(df.info())



=== FINAL INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.8 KB
None


In [87]:
print("\n=== FINAL MISSING VALUES ===")
print(df.isnull().sum())



=== FINAL MISSING VALUES ===
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


In [88]:
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")


Saved cleaned dataset to clean_loan_dataset.csv
